<a href="https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and  load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: A page is scored as an opportunity only if it has real search demand (impressions > 100 for the month). Among those pages, if CTR is below 0.15%, that means people see it but don't click, so it gets flagged as low_ctr_visible_page. If avg_position is worse than 10, the page isn't ranking well despite demand, so it gets flagged as poor_position. If both apply, it's the highest priority opportunity, flagged as low_ctr_and_poor_position. Pages with real demand that don't hit either problem are performing_well and left alone. Pages with low demand (under 100 impressions) are low_demand, meaning there isn't enough traffic yet to justify reviewing them

In [79]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [80]:
monthly = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_impressions) AS total_impressions,
  SUM(gsc_clicks) AS total_clicks,
  AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

monthly.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(321546, 4)

Signal 1: Position vs CTR (ties to the CTR-vs-position flag). Verdict: CONFIRMED.
I grouped pages with more than 100 impressions in February 2026 by average position. CTR fell at every step: 0.41% at positions 1-3 (n = 11,074), 0.34% at 4-10 (n = 40,985), 0.31% at 11-20 (n = 16,706), and 0.15% at 20+ (n = 11,358). This is an observed, directional pattern: pages ranked lower get a smaller share of clicks. This means low CTR is partly explained by position, so my rule’s “low CTR and poor position” group may count one problem twice. The drop from 1-3 to 11-20 is small, though, so position alone does not explain all of the low CTR.

In [81]:
t = monthly[monthly['total_impressions'] > 100].copy()
t['position_bucket'] = pd.cut(t['avg_position'], [0, 3, 10, 20, 10000],
                              labels=['1-3', '4-10', '11-20', '20+'])
sig1 = t.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('total_clicks', 'sum'),
    impressions=('total_impressions', 'sum'))
sig1['ctr_pct'] = sig1['clicks'] / sig1['impressions'] * 100
sig1

,n,clicks,impressions,ctr_pct
position_bucket,,,,
1-3,11074,136800.0,33408697.0,0.409474
4-10,40985,343576.0,102477704.0,0.335269
11-20,16706,69201.0,22252028.0,0.310987
20+,11358,30491.0,20250234.0,0.150571


In [82]:
d = monthly[monthly['total_impressions'] > 100].copy()
d['ctr'] = d['total_clicks'] / d['total_impressions'] * 100
print("share of pages with CTR below 2%:", round((d['ctr'] < 2).mean() * 100, 1), "%")
print(d['ctr'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(3))

share of pages with CTR below 2%: 99.2 %
0.10    0.000
0.25    0.000
0.50    0.139
0.75    0.385
0.90    0.709
Name: ctr, dtype: float64


In [83]:
for cut in [0.1, 0.15, 0.5]:
    print(cut, round((d['ctr'] < cut).mean() * 100, 1), "% of pages flagged")

0.1 44.9 % of pages flagged
0.15 51.3 % of pages flagged
0.5 82.6 % of pages flagged


In [84]:
fair = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_sum_position)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS fair_pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()
chk = monthly.merge(fair, on='content_hash_id')
chk = chk[chk['total_impressions'] > 100]
print("pages whose poor-position flag changes:",
      ((chk['avg_position'] > 10) != (chk['fair_pos'] > 10)).sum(), "of", len(chk))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages whose poor-position flag changes: 2585 of 80124


Signal 2: Impressions volume (ties to the volume / quick-win flag). Verdict: CONFIRMED.
I grouped all 321,546 pages by their February 2026 impressions. The share of pages with at least one click rose steadily: 2.05% at 0-100 impressions (n = 241,422), 41.9% at 101-1,000 (n = 46,837), 90.8% at 1,001-10,000 (n = 29,957), and 99.5% at 10,000+ (n = 3,330). This is an observed pattern: below about 100 impressions, CTR is mostly zeros and too noisy to judge, which supports my rule’s 100-impression cutoff and the low_demand label. The pooled CTR itself stays flat across groups (0.25% to 0.35%), so volume tells me whether a page has enough signal to judge, not whether it is performing well. Some of the rise is expected simply because more impressions give more chances to be clicked.

In [85]:
monthly['volume_bucket'] = pd.cut(monthly['total_impressions'],
                                  [-1, 100, 1000, 10000, 10**9],
                                  labels=['0-100', '101-1000', '1001-10000', '10000+'])
sig2 = monthly.groupby('volume_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('total_clicks', 'sum'),
    impressions=('total_impressions', 'sum'),
    pct_with_a_click=('total_clicks', lambda s: (s > 0).mean() * 100))
sig2['ctr_pct'] = sig2['clicks'] / sig2['impressions'] * 100
sig2

,n,clicks,impressions,pct_with_a_click,ctr_pct
volume_bucket,,,,,
0-100,241422,6137.0,1739654.0,2.050352,0.352771
101-1000,46837,45194.0,18155086.0,41.877148,0.248933
1001-10000,29957,292219.0,91767417.0,90.840204,0.318434
10000+,3330,242655.0,68466765.0,99.489489,0.354413


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [86]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_score = con.sql(f"""
SELECT
  content_hash_id,
  SUM(gsc_impressions) as total_impressions,
  SUM(gsc_clicks) as total_clicks,
  AVG(gsc_avg_position) as avg_position,
  CASE WHEN SUM(gsc_impressions) > 0
    THEN (SUM(gsc_clicks)::FLOAT / SUM(gsc_impressions)) * 100
    ELSE 0 END as ctr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

def score_page(row):
    if row['total_impressions'] <= 100:
        return 0, 'low_demand', 'leave_as_is'

    low_ctr = row['ctr'] < 0.15
    poor_position = row['avg_position'] > 10

    if not low_ctr and not poor_position:
        return 0, 'performing_well', 'leave_as_is'

    severity = 0
    if low_ctr:
        severity += 2
    if poor_position:
        severity += 1

    score = row['total_impressions'] * severity

    if low_ctr and poor_position:
        reason = 'low_ctr_and_poor_position'
        action = 'review_and_refresh'
    elif low_ctr:
        reason = 'low_ctr_visible_page'
        action = 'review_ctr'
    else:
        reason = 'poor_position'
        action = 'review_position'

    return score, reason, action

df_score[['action_score', 'reason_code', 'suggested_action']] = df_score.apply(
    lambda row: pd.Series(score_page(row)), axis=1
)

df_score = df_score.sort_values('action_score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
df_score.to_csv('work/outputs/baseline_action_score.csv', index=False)

df_score.head(20)

,content_hash_id,total_impressions,total_clicks,avg_position,ctr,action_score,reason_code,suggested_action
205227,content_8e1334d6356668e3,203401.0,2.0,4.967059,0.000983,406802.0,low_ctr_visible_page,review_ctr
190539,content_9c057b66c30a3abb,195648.0,1.0,2.488437,0.000511,391296.0,low_ctr_visible_page,review_ctr
44302,content_fec55986a1868d62,193954.0,0.0,3.844678,0.000000,387908.0,low_ctr_visible_page,review_ctr
88951,content_df47d1b976106de4,85935.0,103.0,18.008694,0.119858,257805.0,low_ctr_and_poor_position,review_and_refresh
44301,content_c9f840183215651b,125035.0,0.0,9.366950,0.000000,250070.0,low_ctr_visible_page,review_ctr
6814,content_84a6bf3578312e90,79986.0,74.0,20.050958,0.092516,239958.0,low_ctr_and_poor_position,review_and_refresh
254463,content_7c6373141eae744a,116407.0,101.0,5.129799,0.086765,232814.0,low_ctr_visible_page,review_ctr
8142,content_ba462518dad435fc,69134.0,39.0,27.753576,0.056412,207402.0,low_ctr_and_poor_position,review_and_refresh
280321,content_2ac8c7995de53cd1,92128.0,4.0,6.192190,0.004342,184256.0,low_ctr_visible_page,review_ctr
120250,content_44f34c0a90047651,90223.0,13.0,5.133139,0.014409,180446.0,low_ctr_visible_page,review_ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Row | Impressions | Clicks | Position | Action |
|---|---|---|---|---|
| 1 | 203,401 | 2 | 4.97 | review_ctr |
| 2 | 195,648 | 1 | 2.49 | review_ctr |
| 3 | 193,954 | 0 | 3.84 | review_ctr |
| 4 | 85,935 | 103 | 18.0 | review_and_refresh |
| 5 | 125,035 | 0 | 9.37 | review_ctr |
| 6 | 79,986 | 74 | 20.05 | review_and_refresh |
| 7 | 116,407 | 101 | 5.13 | review_ctr |
| 8 | 69,134 | 39 | 27.75 | review_and_refresh |
| 9 | 92,128 | 4 | 6.19 | review_ctr |
| 10 | 90,223 | 13 | 5.13 | review_ctr |

**Row 1 (content_8e1334d6356668e3):** review_ctr. It has 203,401 impressions at position 4.97 but only 2 clicks (CTR about 0.001%), far below what a top-5 position normally earns. This could be wrong if clicks are not being recorded properly for this page (a tracking gap), or if searchers get their answer on the results page without clicking.

**Row 2 (content_9c057b66c30a3abb):** review_ctr. It has 195,648 impressions at position 2.49 and a single click, a very large gap for such a high position. This could be wrong for the same reasons as Row 1: missing click data, or searches that are answered without a click.

**Row 3 (content_fec55986a1868d62):** review_ctr. It has 193,954 impressions at position 3.84 and zero clicks. This could be wrong if click tracking is missing for this page, so I would check the data before touching the content. My confidence is low and this reading is directional only.

**Row 4 (content_df47d1b976106de4):** review_and_refresh. It has 85,935 impressions at position 18.0 with 103 clicks (CTR about 0.12%), so both problems apply. This could be wrong if low CTR is simply normal on page 2 (my position table shows CTR falling as position worsens), or if the page is new and has not had time to rank.

**Row 5 (content_c9f840183215651b):** review_ctr. It has 125,035 impressions at position 9.37 and zero clicks. This could be wrong if click tracking is missing for this page, or if the average position hides large swings between days (it sits just inside my position-10 cutoff). My confidence is low and this reading is directional only.

**Row 6 (content_84a6bf3578312e90):** review_and_refresh. It has 79,986 impressions at position 20.05 with 74 clicks (CTR about 0.09%), so both problems apply. This could be wrong if low CTR is expected at position 20, or if the page is new and still climbing.

**Row 7 (content_7c6373141eae744a):** review_ctr. It has 116,407 impressions at position 5.13 with 101 clicks (CTR about 0.09%), which is low for a top-5 position. This could be wrong if searchers get their answer on the results page without clicking, or if the average position hides days when the page ranked much lower.

**Row 8 (content_ba462518dad435fc):** review_and_refresh. It has 69,134 impressions at position 27.75 with 39 clicks (CTR about 0.06%), so both problems apply. This could be wrong because position 27 is deep in the results, where low CTR is expected (my position table shows CTR is lowest at 20+), or because the page is new and still climbing.

**Row 9 (content_2ac8c7995de53cd1):** review_ctr. It has 92,128 impressions at position 6.19 with only 4 clicks (CTR about 0.004%), far below what a top-10 position normally earns. This could be wrong if click tracking is missing for this page or if searchers do not need to click. My confidence is low and this reading is directional only.

**Row 10 (content_44f34c0a90047651):** review_ctr. It has 90,223 impressions at position 5.13 with 13 clicks (CTR about 0.01%), which is very low for a top-5 position. This could be wrong for the same reasons as Row 9: missing click data, or searches answered without a click.

In [87]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks**

1. **Rows 1, 2, 3, 5 and 9 (good position, almost no clicks).** These pages have 90,000 to 200,000 impressions and between 0 and 4 clicks at positions 2.5 to 9.4. That is far below what those positions normally earn, so the cause may be a tracking gap or searches answered without a click, not a weak title. They are the picks I trust least, and I would check the data before editing any content.

2. **Rows 4, 6 and 8 (low CTR and poor position).** My position table showed CTR falling as position gets worse, so low CTR at positions 18 to 28 is partly expected. These pages may be behaving normally for their position, and the rule's highest severity may count one problem twice.

3. **The scoring weights.** Score is impressions multiplied by a severity of 1 or 2 (3 when both problems apply). These weights are my own choice, not measured, so the queue mostly ranks the biggest pages that have any problem. This is a transparent baseline, not a tuned one.

**Limits I noticed**
- Average position is a simple mean of daily positions, not weighted by impressions. I tried a weighted version, but it produced impossible positions below 1, so I kept the simple one.
- I set the low-CTR cutoff at 0.15% by looking at February's CTR distribution (it flags about half of pages with over 100 impressions). It was not tested on another month.

**Leakage check**
- **Inputs:** the score uses only total impressions, total clicks and average position, all summed over February 2026.
- **Timing:** the rule runs at the end of a month to plan the next one, so every input is known when the decision is made. No future window is used.
- **No label-derived columns:** the score does not use any outcome or trend column (no `trend_pct` or `trend_direction`).
- **No product flags:** none of FlyRank's own decision flags (such as refresh or CTR-fix flags) are used.

In [88]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.